In [1]:
!uv pip install -q langgraph langchain-openai langchain-anthropic langchain-google-genai langchain-groq langchain-xai deepagents python-dotenv

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from the project environment file
# repository-root .env
env_path = Path("../../.env")
load_dotenv(env_path)

True

In [3]:
######################################################################
## Stream SubAgent Tool Calls
######################################################################
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, AIMessageChunk, AIMessage, ToolMessage, AnyMessage
from deepagents import create_deep_agent, SubAgent

def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)

def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")

def get_weather(city: str) -> str:
    """Get the weather in a given city."""
    return f"The weather in {city} is sunny."

weather_agent = SubAgent(
    name="weather_agent",
    description="Get the weather in a given city.",
    model="openai:gpt-4.1-mini",
    tools=[get_weather],
    system_prompt="You are a helpful assistant"
)

agent = create_deep_agent(
    # model="xai:grok-4",
    model="openai:gpt-4.1-mini",
    subagents=[weather_agent],
    system_prompt="You are a helpful assistant.",
    checkpointer=InMemorySaver()
)

# display(Image(agent.get_graph().draw_mermaid_png()))
input_message = HumanMessage(content="Weather in Dallas?")

current_agent = None
for _, stream_mode, data in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
    config={"configurable": {"thread_id": "subagent-tool-calls"}},
    subgraphs=True,
):
    if stream_mode == "messages":
        token, metadata = data
        if agent_name := metadata.get("lc_agent_name"):
            if agent_name != current_agent:
                print(f"🤖 {agent_name}: ")
                current_agent = agent_name
        if isinstance(token, AIMessage):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):
                _render_completed_message(update["messages"][-1])

[{'name': 'task', 'args': '', 'id': 'call_oQ8gzQVIoc8Gm4Cfy0xw0uC7', 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '{"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'description', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': '":"', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': 'Get', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ' the', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ' current', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ' weather', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ' in', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ' Dallas', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ' including', 'id': None, 'index': 0, 'type': 'tool_call_chunk'}]
[{'name': None, 'args': ' temper